In [0]:
%sql
SELECT * FROM workspace.default.telco_churn LIMIT 10;

customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.3,1840.75,No
9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.7,151.65,Yes
9305-CDSKC,Female,0,No,No,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes
1452-KIOVK,Male,0,No,Yes,22,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,No,Month-to-month,Yes,Credit card (automatic),89.1,1949.4,No
6713-OKOMC,Female,0,No,No,10,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,No,Mailed check,29.75,301.9,No
7892-POOKP,Female,0,Yes,No,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.8,3046.05,Yes
6388-TABGU,Male,0,No,Yes,62,Yes,No,DSL,Yes,Yes,No,No,No,No,One year,No,Bank transfer (automatic),56.15,3487.95,No


In [0]:
%sql
SELECT COUNT(*) AS total_rows FROM workspace.default.telco_churn;

total_rows
7043


In [0]:
%sql
SELECT Churn, COUNT(*) AS customers
FROM workspace.default.telco_churn
GROUP BY Churn;


Churn,customers
No,5174
Yes,1869


In [0]:
%sql
SELECT Churn, COUNT(*) AS customers
FROM workspace.default.telco_churn
GROUP BY Churn;

Churn,customers
No,5174
Yes,1869


In [0]:
%sql
SELECT Contract, Churn, COUNT(*) AS customers
FROM workspace.default.telco_churn
GROUP BY Contract, Churn
ORDER BY Contract;

Contract,Churn,customers
Month-to-month,Yes,1655
Month-to-month,No,2220
One year,Yes,166
One year,No,1307
Two year,No,1647
Two year,Yes,48


In [0]:
import pandas as pd

df = spark.table("workspace.default.telco_churn").toPandas()
print("Loaded:", df.shape)

# 1. TotalCharges is text in this dataset, so convert it to numbers
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# 2. Check missing values
print(df.isnull().sum()[df.isnull().sum() > 0])

# 3. Fill missing values with the median
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

# 4. Convert the target Yes/No to 1/0
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# 5. Drop the ID column (no pattern to learn from)
df = df.drop(columns=["customerID"])

# 6. Convert text columns to 0/1 columns (one-hot encoding)
df = pd.get_dummies(df, drop_first=True)

print("Cleaned:", df.shape)
df.head()

Loaded: (7043, 21)
TotalCharges    11
dtype: int64
Cleaned: (7043, 31)


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,InternetService_Fiber optic,InternetService_No,OnlineSecurity_No internet service,OnlineSecurity_Yes,OnlineBackup_No internet service,OnlineBackup_Yes,DeviceProtection_No internet service,DeviceProtection_Yes,TechSupport_No internet service,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,False,True,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,0,True,False,False,True,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,1,True,False,False,True,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,0,True,False,False,False,True,False,False,False,False,True,False,False,False,True,False,True,False,False,False,False,True,False,False,False,False,False
4,0,2,70.70,151.65,1,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False


In [0]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Churn"]).astype(float)
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

(5634, 30) (1409, 30)


In [0]:
import mlflow, mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

models = {
    "logistic_regression": LogisticRegression(max_iter=2000),
    "random_forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "gradient_boosting": GradientBoostingClassifier(random_state=42),
}

for name, model in models.items():
    with mlflow.start_run(run_name=name):
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        probs = model.predict_proba(X_test)[:, 1]
        acc = accuracy_score(y_test, preds)
        f1 = f1_score(y_test, preds)
        auc = roc_auc_score(y_test, probs)

        mlflow.log_param("model_type", name)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1", f1)
        mlflow.log_metric("roc_auc", auc)
        mlflow.sklearn.log_model(model, "model", input_example=X_train.head(3))
        print(f"{name}: accuracy={acc:.3f}, f1={f1:.3f}, roc_auc={auc:.3f}")

2026/10/31 16:49:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-215c84c2-31ca.cloud.databricks.com/ml/experiments/973050541370099/models/m-90b22eadf505492dae71aef9e9b5abae?o=7474654536042486
/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for m

logistic_regression: accuracy=0.806, f1=0.605, roc_auc=0.842


2026/10/31 16:49:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-215c84c2-31ca.cloud.databricks.com/ml/experiments/973050541370099/models/m-b73d3f1f935341abae51ba8d209f3348?o=7474654536042486
/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for m

random_forest: accuracy=0.790, f1=0.556, roc_auc=0.826


2026/10/31 16:49:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-215c84c2-31ca.cloud.databricks.com/ml/experiments/973050541370099/models/m-0b7cb902de674571968340e02c498b14?o=7474654536042486
/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for m

gradient_boosting: accuracy=0.799, f1=0.577, roc_auc=0.843


In [0]:
mlflow.set_registry_uri("databricks-uc")

runs = mlflow.search_runs(order_by=["metrics.roc_auc DESC"])
best = runs.iloc[0]
print("Best:", best["tags.mlflow.runName"], best["metrics.roc_auc"])

result = mlflow.register_model(
    f"runs:/{best['run_id']}/model", "workspace.default.churn_model"
)
print("Version:", result.version)

Best: gradient_boosting 0.8426257459505542


Successfully registered model 'workspace.default.churn_model'.
2026/10/31 16:53:13 WARNING mlflow.tracking._model_registry.fluent: Run with id a089de16f637449ebfafc9a60c05a932 has no artifacts at artifact path 'model', registering model based on models:/m-0b7cb902de674571968340e02c498b14 instead


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

Version: 1


🔗 Created version '1' of model 'workspace.default.churn_model': https://dbc-215c84c2-31ca.cloud.databricks.com/explore/data/models/workspace/default/churn_model/version/1?o=7474654536042486


In [0]:
import json
print(json.dumps(X.columns.tolist()))

["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges", "gender_Male", "Partner_Yes", "Dependents_Yes", "PhoneService_Yes", "MultipleLines_No phone service", "MultipleLines_Yes", "InternetService_Fiber optic", "InternetService_No", "OnlineSecurity_No internet service", "OnlineSecurity_Yes", "OnlineBackup_No internet service", "OnlineBackup_Yes", "DeviceProtection_No internet service", "DeviceProtection_Yes", "TechSupport_No internet service", "TechSupport_Yes", "StreamingTV_No internet service", "StreamingTV_Yes", "StreamingMovies_No internet service", "StreamingMovies_Yes", "Contract_One year", "Contract_Two year", "PaperlessBilling_Yes", "PaymentMethod_Credit card (automatic)", "PaymentMethod_Electronic check", "PaymentMethod_Mailed check"]


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.model_files;


In [0]:
import pickle, sklearn

model = mlflow.sklearn.load_model("models:/workspace.default.churn_model/1")

with open("/Volumes/workspace/default/model_files/model.pkl", "wb") as f:
    pickle.dump(model, f)

print("scikit-learn version:", sklearn.__version__)

scikit-learn version: 1.7.2


In [0]:
%sql
SELECT * FROM `workspace`.`default`.`telco_churn`;